# Using the BaseIOModeObject Module in baseobjects

## Introduction

The `BaseIOModeObject` module provides a base class for objects that have an I/O mode (e.g., read, write, append) and track whether they are currently open or closed. It also includes several decorators to manage and restrict access to methods based on the object's current state.

This module is particularly useful for building file-like objects, database connections, or any resource that requires explicit state management.

This tutorial covers:
- Creating a subclass of `BaseIOModeObject`
- Implementing open and close operations
- Using context managers for automatic state management
- Restricting method access with `@staterestriction`
- Ensuring objects are open with `@asopen` and `@asopenasync`

**Prerequisites:**
- Basic familiarity with Python
- Understanding of Enums and Context Managers

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)


## Importing the Module

To use the `BaseIOModeObject` and its decorators, import them from the `baseobjects.state` subpackage.


In [ ]:
from enum import StrEnum
from io import UnsupportedOperation
from baseobjects.state import BaseIOModeObject, asopen, staterestriction


## Core Functionality

To use `BaseIOModeObject`, you should subclass it and define the valid modes for your object. You must also implement the abstract `open()` and `close()` methods.

### Key Concepts

1. **_valid_modes**: A class attribute that specifies the `StrEnum` of valid I/O modes.
2. **_is_open**: A boolean flag tracking the open state.
3. **_mode**: The current I/O mode.

### Implementation Example


In [ ]:
class ExampleModes(StrEnum):
    """The modes for the example object. A StrEnum allows for a restricted set of values."""
    READ = "r"
    WRITE = "w"
    APPEND = "a"


class ExampleIOObject(BaseIOModeObject):
    """An example object that has an I/O mode.

    Attributes:
        _valid_modes: The valid modes for this object.
        data: The data of the object.
    """
    _valid_modes = ExampleModes

    def __init__(self, mode: str | StrEnum = ExampleModes.READ):
        """Initializes the example object.

        Args:
            mode: The I/O mode of the object.
        """
        super().__init__()
        self._mode = ExampleModes(mode)
        self.data = "Some initial data."

    def open(self, *args, **kwargs):
        """Opens the object.

        Args:
            *args: Positional arguments for opening.
            **kwargs: Keyword arguments for opening.

        Returns:
            This object.
        """
        print(f"Opening in mode: {self._mode}")
        return super().open(*args, **kwargs)

    def close(self):
        """Closes the object."""
        print("Closing the object.")
        super().close()


# Usage
obj = ExampleIOObject(mode=ExampleModes.READ)
print(f"Is open? {obj.is_open}")
obj.open()
print(f"Is open? {obj.is_open}")
obj.close()
print(f"Is open? {obj.is_open}")


## Module Interaction

### Context Managers

`BaseIOModeObject` supports the context manager protocol. Entering the context automatically opens the object if it's not already open, and exiting the context closes it.


In [ ]:
obj = ExampleIOObject(mode=ExampleModes.READ)

with obj:
    print(f"Inside context - Is open? {obj.is_open}")

print(f"Outside context - Is open? {obj.is_open}")


### The `as_open()` Method

If you want to ensure an object is open for a block of code but only close it if *you* were the one who opened it, use `as_open()`.


In [ ]:
obj = ExampleIOObject(mode=ExampleModes.READ)

with obj.as_open():
    print(f"Ensured open - Is open? {obj.is_open}")


## Advanced Features

### State Restrictions

The `@staterestriction` decorator allows you to enforce that a method can only be called when the object is in a specific state (open or closed) and/or in specific modes.


In [ ]:
class RestrictedObject(ExampleIOObject):
    """An object with restricted method access."""
    @staterestriction(open_state=True, valid_modes=ExampleModes.READ)
    def read_data(self):
        """Reads the data."""
        return self.data

    @staterestriction(open_state=True, valid_modes={"w", "a"})
    def write_data(self, value):
        """Writes the data.

        Args:
            value: The data to write.
        """
        self.data = value


obj = RestrictedObject(mode=ExampleModes.READ)

# This will fail because the object is closed
try:
    obj.read_data()
except ValueError as e:
    print(f"Error: {e}")

# This will fail because the mode is READ, but we are trying to write
with obj:
    try:
        obj.write_data("New Data")
    except UnsupportedOperation as e:
        print(f"Error: {e}")


### Automatic Opening with `@asopen`

The `@asopen` decorator ensures that the object is open for the duration of the method call. It uses the `as_open()` context manager internally.


In [ ]:
class AutoOpenObject(ExampleIOObject):
    """An object that automatically opens when a method is called."""
    @asopen()
    def process(self):
        """Processes the data."""
        print(f"Processing data: {self.data}")


obj = AutoOpenObject()
# Method handles opening and closing automatically
obj.process()
print(f"Is open after process? {obj.is_open}")


## Examples

### Complete Workflow

This example demonstrates a typical workflow using `BaseIOModeObject` with state-restricted methods.


In [ ]:
class FileManager(BaseIOModeObject):
    """A mock file manager.

    Attributes:
        _valid_modes: The valid modes for this object.
        content: The content of the file.
    """
    _valid_modes = ExampleModes

    def __init__(self, mode: str | StrEnum = ExampleModes.READ):
        """Initializes the file manager.

        Args:
            mode: The I/O mode of the object.
        """
        super().__init__()
        self.mode = mode
        self.content = ""

    def open(self):
        """Opens the file."""
        print(f"--- File Opened in {self.mode} mode ---")
        return super().open()

    def close(self):
        """Closes the file."""
        print("--- File Closed ---")
        super().close()

    @staterestriction(open_state=True, valid_modes="r")
    def read(self):
        """Reads the file content."""
        return self.content

    @staterestriction(open_state=True, valid_modes=ExampleModes.WRITE)
    def write(self, text):
        """Writes the file content.

        Args:
            text: The content to write.
        """
        self.content = text


# Writing to "file"
writer = FileManager(mode="w")
with writer:
    writer.write("Hello, BaseIOModeObject!")

# Reading from "file"
reader = FileManager(mode=ExampleModes.READ)
reader.content = writer.content  # Simulate shared storage
with reader:
    print(f"Read content: {reader.read()}")


## API Highlights

- **`BaseIOModeObject`**: The abstract base class.
    - `is_open`: Property returning the open state.
    - `mode`: Property for the current I/O mode.
    - `open(*args, **kwargs)`: Abstract method to open the object.
    - `close()`: Abstract method to close the object.
    - `as_open(*args, **kwargs)`: Context manager that ensures the object is open.
    - `require_open()`: Raises `ValueError` if the object is closed.
    - `require_mode(modes)`: Raises `UnsupportedOperation` if the mode is invalid.
- **`staterestriction(open_state=None, valid_modes=None)`**: Decorator to restrict method access.
- **`asopen(*args, **kwargs)`**: Decorator that wraps a method call in an `as_open()` context.
- **`asopenasync(*args, **kwargs)`**: Asynchronous version of `@asopen`.


## Troubleshooting / FAQs

- **Problem**: `ValueError: Operation on a closed ExampleIOObject is not allowed.`
    - **Solution**: Ensure you are calling the method within a `with obj:` block or call `obj.open()` beforehand. Alternatively, use the `@asopen` decorator.
- **Problem**: `UnsupportedOperation: Mode r is not valid for this operation.`
    - **Solution**: The method you are calling is restricted to certain modes. Check the object's `mode` attribute and ensure it matches the required modes for the method.


## Conclusion and Next Steps

In this tutorial, you learned how to use `BaseIOModeObject` to manage the state and I/O modes of your objects. You also explored how to use decorators like `@staterestriction` and `@asopen` to simplify state management and enforce correct usage.

- **Next**: Check out the `BaseObject` tutorial to learn more about the foundation of this package.
- **Reference**: See `src/baseobjects/state/baseiomodeobject.py` for the full implementation.
